# Simulating robot systems with MuJoCo

This notebook introduces the MuJoCo-based simulation workflow in RobotBlockSet. It shows how to connect RobotBlockSet robot objects to MuJoCo scenes, create simulated robots, and run motion-related examples.

## What this notebook covers

The examples demonstrate how RobotBlockSet connects robot models and scenes to MuJoCo, how robot objects are created for the available MuJoCo backends, and how the resulting setup can be used for motion execution, visualization, sensing, and controller experiments.

Use this notebook as a practical starting point when you want to run robot-system simulations with MuJoCo or adapt the workflow to your own MJCF models and simulated environments.

# Imports

The first code cell imports NumPy, plotting utilities, RobotBlockSet path helpers, and the MuJoCo classes used in the examples.

In [1]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from datetime import datetime

from robotblockset.tools import get_rbs_path

np.set_printoptions(formatter={"float": "{: 0.4f}".format})

# Robot classes for the MuJoCo simulator

MuJoCo, short for Multi-Joint dynamics with Contact, is a high-performance physics engine used for robotics, biomechanics, and reinforcement learning. It provides efficient simulation of articulated mechanisms, contacts, actuators, and sensors. Robot models in MuJoCo are described with MJCF, an XML-based modeling language that defines kinematics, dynamics, actuators, sensors, and scene objects.

RobotBlockSet integrates MuJoCo in three ways:

1. **mujoco** - standalone MuJoCo application. RobotBlockSet communicates with `simmujoco`, an extended version of MuJoCo's `simulate` application with a TCP server. This interface can read robot states and sensor data, access object poses, control actuators, and reposition models in the scene. To build `simmujoco`, see `robotblockset/mujoco/simmujoco/README.md`.
2. **pymujoco** - MuJoCo with native Python bindings. RobotBlockSet interacts directly with the MuJoCo engine and viewer through Python bindings, while the simulator runs independently in a separate thread.
3. **pymujoco_sim** - RobotBlockSet-controlled MuJoCo integration with native Python bindings. RobotBlockSet manages simulation stepping, which enables precise synchronization and supports headless simulation.

These options let RobotBlockSet cover use cases from interactive visual debugging to automated, synchronized simulation workflows.

MuJoCo robot models are defined in MJCF XML files. For details, see the MuJoCo modeling documentation: https://mujoco.readthedocs.io/en/stable/modeling.html. The `tutorial_generate_MJCF_scene.ipynb` notebook shows how to generate complex MJCF scenes programmatically.

MuJoCo-specific parameters and methods are implemented in these subclasses:

- `robots_mujoco.py` for the **mujoco** target,
- `robots_pymujoco.py` for the **pymujoco** target,
- `robots_pymujoco_sim.py` for the **pymujoco_sim** target.

To control a robot model in MuJoCo, RobotBlockSet sends commands to actuators and reads state or sensor data from the simulator. Current MuJoCo robot targets assume that all robot joints are connected to actuators with internal position control, so these targets support the joint-position control strategy.

> **Note:** Custom control algorithms can be implemented with the `robot_pymujoco.py` and `robot_pymujoco_sim.py` targets. In that case, sensors and actuators must be defined appropriately in the MJCF model, and the custom controller should send commands through the available MuJoCo actuator interfaces.

# Creating a robot object

The way you create a robot object depends on the selected MuJoCo backend.

## Target **mujoco**

The **mujoco** target uses `simmujoco`, an extended version of MuJoCo's `simulate` application.

First, load the corresponding MJCF model into `simmujoco`. Then create the RobotBlockSet robot object that connects to that simulator instance.

### Manual opening of the scene

If you open the `simmujoco` application directly, load the MJCF model by dropping the corresponding model file (`<model>.xml`) into the simulator window.

In [14]:
from robotblockset.mujoco.robots_mujoco import panda

r = panda(robot_name="panda")

[RBS_INFO] [1773401731.547704935] [panda_MuJoCo]: Robot connected to MuJoCo


> **Warning:** The `robot_name` must match the name of the first body in the robot body tree.

### Programmatically loading the scene

Another option is to start the simulator from Python. The example below starts MuJoCo on **localhost** with a non-default **port**, which avoids connecting to another simulator instance that may already be open.

> **Note:** The path to `simmujoco` in the `subprocess.Popen(...)` call must match your local installation. If `simmujoco` was built or installed in a different location, update the path accordingly. The example below uses the executable from the local RobotBlockSet folder returned by `get_rbs_path()`.

In [3]:
import subprocess

p = subprocess.Popen([get_rbs_path() + "/mujoco/simmujoco/bin/win64/bin/simmujoco.exe", "localhost", "50001"])

Open a connection to the simulator by creating a scene object for the selected port:

In [4]:
from robotblockset.mujoco.mujoco_api import mjInterface as mujoco_scene

scene = mujoco_scene(port=50001)
scene.mj_connect()

0

Load the MJCF scene into the MuJoCo simulator:

In [5]:
from time import sleep

scene.mj_load(get_rbs_path() + "/mujoco/mjcf_models/panda_scene.xml")
sleep(2)

Finally, create the robot object:

In [6]:
from robotblockset.mujoco.robots_mujoco import panda

r = panda(robot_name="panda", scene=scene)

[RBS_INFO] [16:56:10] [panda_MuJoCo]: Robot connected to MuJoCo


## Target **pymujoco** or **pymujoco_sim**

For the **pymujoco** and **pymujoco_sim** targets, first create a MuJoCo scene object.

In [2]:
from robotblockset.mujoco.robots_pymujoco import mujoco_scene, panda

MODEL_PATH = get_rbs_path() + "/mujoco/mjcf_models/panda_scene.xml"
scene = mujoco_scene(MODEL_PATH, show_camera=None, verbose=1)

Arguments for `mujoco_scene`:

| Argument | Default | Description |
| --- | --- | --- |
| `model_xml_file` | `None` | Path to the MuJoCo model XML file to load. |
| `model` | `None` | Existing `mujoco.MjModel` instance; used instead of loading from XML. |
| `show_viewer` | `True` | If `True`, open the viewer. This argument is used by **pymujoco_sim**. |
| `show_camera` | `None` | List of camera IDs or names to display in separate windows; `None` opens no extra camera windows. |
| `synchronized` | `True` | If `True`, synchronize simulation steps to a fixed timestep. This argument is used by **pymujoco**. |
| `verbose` | `0` | Verbosity level for debug messages. |

Then define the robot object:

In [ ]:
r = panda(scene=scene)

[RBS_INFO] [11:00:16.333] [panda_PyMuJoCo]: Robot connected to MuJoCo


Arguments for `robot_mujoco` / `robot_pymujoco` across `robots_mujoco.py`, `robots_pymujoco.py`, and `robots_pymujoco_sim.py`:

| Argument | Default | Description |
| --- | --- | --- |
| `robot_name` | required | Robot name used for naming and model lookups. |
| `scene` | `None` | MuJoCo scene or connection object. It is required for `robot_pymujoco` and optional for `robot_mujoco`, which can create its own interface if `scene` is `None`. |
| `host` | `"localhost"` | MuJoCo server host; used only by `robot_mujoco`. |
| `port` | `50000` | MuJoCo server port; used only by `robot_mujoco`. |
| `**kwargs` | - | Additional keyword arguments, including model-handle names described below. |

Move to homeconfiguration

In [8]:
r.JMove(r.q_home)

0

## Definition of model handles

To read states and sensors, and to send control signals, the backend needs the indices or handles of MuJoCo objects.

The simplest approach is to name MJCF objects according to the RobotBlockSet naming convention. The object name is built from the robot name `<robot>`, followed by `_`, followed by a descriptive object name, and in some cases a numeric suffix.

| Property | Default MuJoCo name | Description |
| --- | --- | --- |
| `robot_name` | `<robot>` | Base link name, that is, the first body in the MuJoCo robot body tree. This defines the robot base frame. |
| `JointNames` | `<robot>_joint<i>` | Joint names in the MuJoCo robot model. |
| `ActuatorNames` | `<robot>_actuator<i>` | Actuator names for joint-position control in the MuJoCo robot model. |
| `FlangeName` | `<robot>_flange` | Site attached to the last link, representing the end-effector flange. |
| `TCPName` | `<robot>_TCP` | Site at the TCP location. |
| `SensorJointPosName` | `<robot>_pos_joint<i>` | Joint-position sensor names in the MuJoCo model. |
| `SensorJointVelName` | `<robot>_vel_joint<i>` | Joint-velocity sensor names in the MuJoCo model. |
| `SensorPosName` | `<robot>_pos` | End-effector position sensor name in the MuJoCo model. |
| `SensorOriName` | `<robot>_ori` | End-effector orientation sensor name in the MuJoCo model. |
| `SensorLinVelName` | `<robot>_v` | End-effector linear velocity sensor name in the MuJoCo model. |
| `SensorRotVelName` | `<robot>_w` | End-effector angular velocity sensor name in the MuJoCo model. |
| `SensorForceName` | `<robot>_force` | End-effector force sensor name in the MuJoCo model. |
| `SensorTorqueName` | `<robot>_torque` | End-effector torque sensor name in the MuJoCo model. |

> **Note:** Joint positions and velocities are updated by default from sensors defined by `SensorJointPosName` and `SensorJointVelName`. If these sensors are not defined in the model, joint positions and velocities are read from the model state using indices obtained from `JointNames`.

A typical robot body definition in an MJCF model looks like this:
```xml
<body name="panda" childclass="panda_robot">
  <body name="panda_link0">
    <inertial pos="-0.041018 -0.00014 0.049974" quat="0.00630474 0.751245 0.00741774 0.659952" mass="0.629769" diaginertia="0.00430465 0.00387984 0.00313051"/>
    <geom  ... />
    <body name="panda_link1" pos="0 0 0.333">
      <inertial pos="0.003875 0.002081 -0.04762" quat="0.711549 0.00634377 -0.0131124 0.702485" mass="4.97068" diaginertia="0.707137 0.703435 0.00852456"/>
      <joint name="panda_joint1" pos="0 0 0" axis="0 0 1"/>
      <geom ... />
      <body name="panda_link2" quat="0.707107 -0.707107 0 0">
      ...
      </body>
    </body>
  </body>
</body>
```

A typical actuator definition in an MJCF model looks like this:

```xml
<actuator>
  <general name="panda_actuator1" class="panda_robot" joint="panda_joint1" gainprm="4500" biasprm="0 -4500 -450"/>
  <general name="panda_actuator2" class="panda_robot" joint="panda_joint2" ctrlrange="-1.7628 1.7628" gainprm="4500" biasprm="0 -4500 -450"/>
  <general name="panda_actuator3" class="panda_robot" joint="panda_joint3" gainprm="3500" biasprm="0 -3500 -350"/>
  <general name="panda_actuator4" class="panda_robot" joint="panda_joint4" ctrlrange="-3.0718 -0.0698" gainprm="3500" biasprm="0 -3500 -350"/>
  <general name="panda_actuator5" class="panda_robot" joint="panda_joint5" forcerange="-12 12" gainprm="2000" biasprm="0 -2000 -200"/>
  <general name="panda_actuator6" class="panda_robot" joint="panda_joint6" ctrlrange="-0.0175 3.7525" forcerange="-12 12" gainprm="2000" biasprm="0 -2000 -200"/>
  <general name="panda_actuator7" class="panda_robot" joint="panda_joint7" forcerange="-12 12" gainprm="2000" biasprm="0 -2000 -200"/>
  <general name="panda_tool_panda_hand" class="panda_tool_panda_hand" tendon="panda_tool_split" ctrlrange="0 255" forcerange="-100 100" gainprm="0.0156863" biasprm="0 -100 -10"/>
</actuator>
```

A typical sensor definition in an MJCF model looks like this:

```xml
<sensor>
	<framepos name="panda_pos" objtype="site" objname="panda_hand" />
	<framequat name="panda_ori" objtype="site" objname="panda_hand" />
	<framelinvel name="panda_v" objtype="site" objname="panda_hand" />
	<frameangvel name="panda_w" objtype="site" objname="panda_hand" />
	<force name="panda_force" site="panda_hand" noise="0.5" />
	<torque name="panda_torque" site="panda_hand" noise="0.5" />
</sensor>
```

MuJoCo robot objects use the same high-level motion API as other RobotBlockSet robots. For example, the following commands generate equivalent trapezoidal joint motion:

```python
r.JMove(q, t, traj="Trap")
r.JLine(q, t)
```

If the MuJoCo robot model does not use the default object names, there are several alternatives:

- Pass the object names when creating the robot object with the corresponding keyword arguments.

```python
r = panda(robot_name="panda", TCPName="panda_gripper")
```

- Define the object names in the robot specification class.

```python
class ur10_spec(robot):
    def __init__(self) -> None:
        self.Name = "ur10"
        self.nj = 6
        ...
        self.joint_names = [
            "shoulder_pan_joint",
            "shoulder_lift_joint",
            "elbow_joint",
            "wrist_1_joint",
            "wrist_2_joint",
            "wrist_3_joint",
        ]
```

- When using **pymujoco** or **pymujoco_sim**, use the constructor arguments `JointNames` and `ActuatorNames` to discover or generate the required names directly from the MJCF model.

Meaning of `JointNames`:

- `JointNames="auto"` searches the MJCF model under the robot base body and automatically collects the first `nj` robot joints from the model tree. This is useful when the model uses custom joint names and you want to read them from the loaded MJCF model.
- `JointNames="gen"` does not inspect the MJCF model for joint names. It generates them with the default naming rule `<robot>_joint1`, `<robot>_joint2`, ..., `<robot>_joint<nj>`.
- `JointNames=None` first tries to use `self.joint_names` from the robot specification class. If these names are not available, it falls back to the generated default names `<robot>_joint<i>`.

Meaning of `ActuatorNames`:

- `ActuatorNames="auto"` finds actuators directly from the MJCF model for the selected robot joints.
- `ActuatorNames=None` generates actuator names from `JointNames` with the default actuator naming rule `<robot>_actuator<i>`.
- These backends do not define an `ActuatorNames="gen"` keyword. Automatic discovery is available through `"auto"`; otherwise actuator names are generated from `JointNames` or given explicitly as a list.

Typical examples:

```python
r = panda(scene=scene, JointNames="auto", ActuatorNames="auto")
```

This is the most robust option when the MJCF model already contains the correct joint and actuator definitions and you want the backend to read them directly from the model.

```python
r = panda(scene=scene, JointNames="gen")
```

This is useful when the generated scene follows the standard naming convention `<robot>_joint<i>`. In this case, actuator names are generated as `<robot>_actuator<i>` by default.

```python
r = ur10e(scene=scene)
```

In this case, `JointNames` is not given explicitly, so the backend uses the names defined in the robot specification class if available.

In short, for **pymujoco** and **pymujoco_sim**:

- use `"auto"` when names should be read from the MJCF model,
- use `"gen"` only for `JointNames` when the standard generated joint naming rule is used,
- use explicit lists or robot specifications when the model uses a custom naming scheme.

A typical sensor definition in an MJCF model looks like this:

```xml
<sensor>
  <framepos name="panda_pos" objtype="site" objname="panda_hand" />
  <framequat name="panda_ori" objtype="site" objname="panda_hand" />
  <framelinvel name="panda_v" objtype="site" objname="panda_hand" />
  <frameangvel name="panda_w" objtype="site" objname="panda_hand" />
  <force name="panda_force" site="panda_hand" noise="0.5" />
  <torque name="panda_torque" site="panda_hand" noise="0.5" />
</sensor>
```

# MuJoCo-specific robot methods

The `robots_mujoco.py`, `robots_pymujoco.py`, and `robots_pymujoco_sim.py` classes define MuJoCo-specific robot methods.

| Method | Description |
| --- | --- |
| `SendRobot_u(u)` | Send joint commands to robot actuators. |
| `SendCtrl(u)` | Send a control vector to all actuators. |
| `SendAuxCtrl(idx, val)` | Update selected actuator controls by index. |
| `GetAuxJointPos(idx)` | Return joint positions for auxiliary joints by index. |
| `GetSensor(name_or_id=None)` | Read sensor data by name or ID, or return the full sensor array. |
| `GetContacts(name_or_id=None)` | Return contact forces for a geom or for all contacts. |
| `GetMocapPose(name_or_id, out="x")` | Return a mocap body pose in the requested output format. |
| `SetRobotPose(x)` | Set the robot pose. |
| `SetMocapPose(name_or_id, x)` | Set the pose of a mocap body. |
| `GetObjectData(name_or_id)` | Return raw MuJoCo body data for a body name or ID. |
| `GetObjectPose(typ, name_or_id, out="x")` | Return the pose of a body, site, or geom in the requested format. |
| `SetObjectPose(name_or_id, x)` | Set a body pose. |
| `SetEquality(name_or_id, active)` | Set an equality-constraint activation flag. |
| `UpdateRobotBaseFromModel()` | Update the cached robot base pose from the MuJoCo model. |
| `SimulatorMessage(msg)` | Send a message to the simulator UI. |
| `sim(dt)` | Advance the simulator for a fixed duration. |

The `robots_pymujoco.py` and `robots_pymujoco_sim.py` classes also define these properties:

| Property | Description |
| --- | --- |
| `J` | Jacobian matrix `J(q)`. |
| `H` | Inertia matrix `H(q)` in joint space. |
| `M` | Inertia matrix `M(q)` in task space. |
| `c` | Coriolis, centrifugal, and gravitational joint torques. |

To compute the Jacobian matrix from the RobotBlockSet kinematic model for a chosen configuration, use:

In [4]:
print("Jacobian matrix using RBS model:\n", r.Jacobi(r.q_home))

Jacobian matrix using RBS model:
 [[ 0.0000  0.1539  0.0000  0.1279  0.0000  0.2104  0.0000]
 [ 0.3069  0.0000  0.3258  0.0000  0.2104  0.0000 -0.0000]
 [ 0.0000 -0.3069  0.0000  0.4720  0.0000  0.0880  0.0000]
 [ 0.0000 -0.0000 -0.7071  0.0000  1.0000  0.0000  0.0000]
 [ 0.0000  1.0000 -0.0000 -1.0000  0.0000 -1.0000  0.0000]
 [ 1.0000  0.0000  0.7071  0.0000 -0.0000  0.0000 -1.0000]]


To read the Jacobian from the MuJoCo model, use:

In [ ]:
print("Jacobian matrix from MuJoCo model:\n", r.J)

Jacobian matrix from MuJoCo model:
 [[ 0.0000  0.1518  0.0000  0.1309 -0.0000  0.2115  0.0000]
 [ 0.3063  0.0000  0.3245  0.0001  0.2111  0.0001  0.0000]
 [ 0.0000 -0.3063  0.0000  0.4698 -0.0001  0.0854 -0.0000]
 [ 0.0000 -0.0000 -0.7031  0.0001  1.0000  0.0001 -0.0125]
 [ 0.0000  1.0000 -0.0000 -1.0000  0.0001 -1.0000 -0.0005]
 [ 1.0000  0.0000  0.7111  0.0001 -0.0050  0.0005 -0.9999]]


> **Note:** To read properties from the MuJoCo model at a desired configuration, move the robot to that configuration first.

To read the robot inertia matrix in task space, use:

In [6]:
print("Inertia matrix intask space:\n", r.M)

Inertia matrix intask space:
 [[ 12.5160 -0.1941  1.5324  0.0432  2.4669  0.0001]
 [-0.1941  5.1477 -0.0339 -0.8114 -0.0440 -0.1925]
 [ 1.5324 -0.0339  6.5362  0.0668  0.8839 -0.0017]
 [ 0.0432 -0.8114  0.0668  0.2530  0.0140  0.0353]
 [ 2.4669 -0.0440  0.8839  0.0140  0.6858 -0.0017]
 [ 0.0001 -0.1925 -0.0017  0.0353 -0.0017  0.0998]]


A mocap body can be positioned at a specific pose in the model. For example, many RobotBlockSet models define a dummy body named **Target**. You can move it to a specific location with:

In [7]:
r.SetMocapPose("Target", [0.5, 0.2, 0.6])

> **Note:** MuJoCo objects can be assigned to visibility groups from `0` to `5`. If you do not see an object in the simulator, check the group visibility settings.

> **Note:** With `robots_pymujoco.py` and `robots_pymujoco_sim.py`, many model and object parameters can be read or set directly. For more details, see the MuJoCo Python documentation: https://mujoco.readthedocs.io/en/stable/python.html